### Import packages
#### Create connection to Database

In [1]:
from sqlalchemy import create_engine
import pandas as pd

# Connect to the databasehttps://github.com/ThaoNhiThan/data_engineering_practices
db_connection_string = 'sqlite:///chinook.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()

#### Read table from database

In [ ]:
# HINTS
# Load data into DataFrame
# user Pandas to read data from a table into a DataFrame


In [2]:
#Read all tables in database
df = pd.read_sql("SELECT Name FROM sqlite_master WHERE type='table';", db_conn)
df


,name
0,albums
1,sqlite_sequence
2,artists
3,customers
4,employees
5,genres
6,invoices
7,invoice_items
8,media_types
9,playlists


In [ ]:
# # Approach 1: Use Pandas.read_sql_table to read all columns from 'customers' table
table_name = 'customers'
# df = pd.read_sql_table()
df = pd.read_sql_table(table_name, con=db_conn)
df.head(5)
df.to_csv('customers.csv', index=False)

In [ ]:
#Import table Albums and export to csv
df = pd.read_sql_table('albums', con=db_conn)
df.tail(5)
df.to_csv('albums.csv', index=False)

In [ ]:
#Import table Artists and export to csv
df = pd.read_sql_table('artists', con=db_conn)
df.tail(5)
df.to_csv('artists.csv', index=False)

In [6]:
# # Approach 2: Use Pandas.read_sql_query to read these columns
# table_name = 'customers'
# columns = ['CustomerId', 'FirstName', 'LastName', 'Phone', 'Email', 'SupportRepId']
df = pd.read_sql_query(sql='select CustomerId, FirstName, LastName from customers', con=db_conn)
df.tail(5)

,CustomerId,FirstName,LastName
54,55,Mark,Taylor
55,56,Diego,Gutiérrez
56,57,Luis,Rojas
57,58,Manoj,Pareek
58,59,Puja,Srivastava


In [9]:
#all tables
import os
folder_path = 'destination/chinook'
#Create this folder… and don’t crash if it already exists
os.makedirs(folder_path, exist_ok=True)
all_tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' and name not like 'sqlite_%';", db_conn)

for table_name in all_tables['name']:
    print(f'Extracting {table_name}...')
    df = pd.read_sql_table(table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv', index=False)
    print('Completed!\n')

Extracting albums...
Completed!

Extracting artists...
Completed!

Extracting customers...
Completed!

Extracting employees...
Completed!

Extracting genres...
Completed!

Extracting invoices...
Completed!

Extracting invoice_items...
Completed!

Extracting media_types...
Completed!

Extracting playlists...
Completed!

Extracting playlist_track...
Completed!

Extracting tracks...
Completed!



#### Config-Driven Ingestion

In [13]:
# # HINTS
# # Read configs stored in the 'config.yml' file

# # Read yaml file
# # Package: yaml (pip install pyyaml)
# # Function: load / safe_load
# # Print it after loading

# import yaml
# import json

# config_file = 'config.yml'
import yaml
import io
config_file = 'config.yml'
f = open(config_file, 'r')
config = yaml.safe_load(f)
config

{'source': {'database': {'name': 'chinook', 'type': 'sqlite'},
  'table': ['albums',
   'artists',
   'customers',
   'employees',
   'genres',
   'invoice_items',
   'invoices',
   'media_types',
   'playlist_track',
   'playlists',
   'tracks']}}

In [17]:
# Use loop function to read tables within config.source.table
# Export output into CSV
# Name Convention: '<date>__<table_name>.csv'
# Path: chinook/config_driven/
# note: use os.makedirs() if path is not exists
def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=con)
    df.to_csv(f'{folder_path}/20251127_{table_name}.csv',index=False)
    print('Completed!\n')

def get_connection(type=None, name=None, **kwargs):
    if type == 'sqlite':
        db_connection_string = f"sqlite:///{name}.db"
        engine = create_engine(db_connection_string)
        return engine.connect()

    elif type == 'Oracle':
        db_connection_string = f"Oracle://{name}:1234"
        engine = create_engine(db_connection_string)
        return engine.connect()

    else:
        raise ValueError(f"Unsupported database type: {type}")
db_conn = get_connection(**config['source']['database'])

for table_name in config['source']['table']:
    extract_table(table_name, db_conn, 'destination/config_driven')

Extracting albums ...
Completed!

Extracting artists ...
Completed!

Extracting customers ...
Completed!

Extracting employees ...
Completed!

Extracting genres ...
Completed!

Extracting invoice_items ...
Completed!

Extracting invoices ...
Completed!

Extracting media_types ...
Completed!

Extracting playlist_track ...
Completed!

Extracting playlists ...
Completed!

Extracting tracks ...
Completed!



### Metadata-Driven Ingestion

In [18]:
# # HINTS
# # Read metadata from the database inlcuding tables / columns
# sqlite_metadata_table = 'sqlite_master'
# sqlite_metadata_condition = "type = 'table'"
# metadata_sql = f""" select 1"""
# print(metadata_sql)
# table_df = pd.read_sql_query(metadata_sql)
# print(table_df)

metadata_sql = """select name from sqlite_master where 1=1 and type = 'table'"""
table_df = pd.read_sql_query(metadata_sql, con=db_conn)
names = [tb for tb in list(table_df['name']) if tb not in ('sqlite_stat1', 'sqlite_sequence')]
import os 


# loop for each table from the DataFrame
# read and extract table
# save to path: chinook/metadata_driven/
# note: use os.makedirs() if path is not exists
def extract_table(table_name, con, folder_path):
    os.makedirs(folder_path, exist_ok=True)
    print(f'Extracting {table_name} ...')
    df = pd.read_sql_table(table_name=table_name, con=db_conn)
    df.to_csv(f'{folder_path}/{table_name}.csv', index=False)
    print('Completed!\n')
for name in names:
    extract_table(table_name=name, con=db_conn, folder_path='destination/metadata')


Extracting albums ...
Completed!

Extracting artists ...
Completed!

Extracting customers ...
Completed!

Extracting employees ...
Completed!

Extracting genres ...
Completed!

Extracting invoices ...
Completed!

Extracting invoice_items ...
Completed!

Extracting media_types ...
Completed!

Extracting playlists ...
Completed!

Extracting playlist_track ...
Completed!

Extracting tracks ...
Completed!



In [ ]:
# loop for each table from the DataFrame
# read and extract table
# save to path: destination/metadata_driven/
# note: use os.makedirs() if path is not exists

In [20]:
#Inspect
from sqlalchemy import inspect
import os
import pandas as pd

# Inspect DB
db_inspector = inspect(db_engine)

tables = db_inspector.get_table_names()
print("Tables found:", tables)

# Prepare output folder
output_folder = 'destination/inspect'
os.makedirs(output_folder, exist_ok=True)

# Loop through tables and extract each one
for table_name in tables:
    print(f"Extracting {table_name} ...")

    # Read table into pandas DataFrame
    df = pd.read_sql_table(table_name, con=db_conn)

    # Create full file path
    file_path = os.path.join(output_folder, f"{table_name}.csv")

    # Save CSV
    df.to_csv(file_path, index=False)

    print("Completed!\n")



Tables found: ['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'invoices', 'media_types', 'playlist_track', 'playlists', 'tracks']
Extracting albums ...
Completed!

Extracting artists ...
Completed!

Extracting customers ...
Completed!

Extracting employees ...
Completed!

Extracting genres ...
Completed!

Extracting invoice_items ...
Completed!

Extracting invoices ...
Completed!

Extracting media_types ...
Completed!

Extracting playlist_track ...
Completed!

Extracting playlists ...
Completed!

Extracting tracks ...
Completed!

